In [21]:
### NAIVE VERSION

In [22]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>

// Neceessary library
#include <cuda_runtime.h>

// --- CUDA Kernel Naive ---
// Global function: callable from Host (CPU), executed on Device (GPU)
__global__ void matMulNaive(const float * a, const float * b, float * c, int n) {

    // Thread mapping: calculating global 2D coordinates for the output matrix C
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Boundary check
    if (row < n && col < n) {
        float sum = 0.0f;
        // Matrix multiplication: dot product by linearizing 2D indices to access 1D flat arrays
        for (int k = 0; k < n; ++k) {
            sum += a[row * n + k] * b[k * n + col];
        }
        // Output
        c[row * n + col] = sum;
    }
}


int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);


    // Host Memory Allocation (1D Flattening)
    size_t bytes = (size_t)n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy data from host to device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel Launch: grid and block dimensions
    // - 1024 threads per Block
    // - Total numbers of block necessary to compute the whole C matrix n x n
    int blockSize = 32;
    dim3 threadsPerBlock(blockSize, blockSize);
    dim3 numBlocks((n + blockSize - 1) / blockSize, (n + blockSize - 1) / blockSize);

    printf("Starting the computation...\n");

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Launch of the kernel + timing
    cudaEventRecord(start);
    matMulNaive<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaEventRecord(stop);

    // Wait for the kernel to finish
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy data from device to host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample_size = (n < 1000) ? n : 1000;
        for (int i = 0; i < sample_size; i++) {
            for (int j = 0; j < sample_size; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    // Memory Deallocation: both Host and Device
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_naive.cu', 'w') as f:
    f.write(cell_str)

In [23]:
### OPTIMIZED VERSION

In [24]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Configuration
// - BM, BN: size of the tile, work for each Thread Block
// - BK: step size, how many elements are loaded together for the multiplication
// - TM, TN: size of the block computed by each single thread
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8


// ---- KERNEL CUDA: 2D Register Tiling ----
__global__ void MatMult(int n, const float *A, const float *B, float *C) {

    // Shared Memory exploited by the Thread Block for the computation of the matrix slice
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    // Local coordinates
    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    // Registers to store the partial results of the multiplication, in order to maximize the speed
    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    // Linear index of the thread for Decoupled Loading
    int tid = ty * (BN / TN) + tx;

    // Loop: slicing along the k dimension (BK=16)
    for (int p = 0; p < (n + BK - 1) / BK; ++p) {

        // Decoupled Loading, each thread transfers 8 elements ((128x16)/256) from the Global Memory to the Shared Memory Tile A and B in a contiguous way
        // - Linear index idx (Memory Coalescing)
        // - a_row, a_col: 2D coordinates within the Shared Memory
        // - g_a_row, g_a_col: transformation in global coordinates
        // - Saving in Shared Memory sA of the value or padding (0) if out of the matrix borders

        // Loading of a slice of the Tile A
        for (int i = 0; i < (BM * BK) / 256; ++i) {
            int idx = i * 256 + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        // Loading of a slice of the Tile B
        for (int i = 0; i < (BK * BN) / 256; ++i) {
            int idx = i * 256 + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }

        // Synchronization Barrier: ensures that all threads have finished loading the data before to start the computation
        __syncthreads();


        // Register Promotion
        // - Moving data from Shared Memory to registers for the computation
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }


    // Saving in the Global Memory C
    // - Each thread saves its 64 results stored in the Registers to the Global Memory
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Host Memory Allocation (1D Flattening)
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Copy from Host to Device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel launch: Grid and Block dimensions
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    printf("Starting the computation...");

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Time Measurement
    cudaEventRecord(start);
    MatMult<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);
    cudaEventRecord(stop);

    // Synchronization: Host waits for Device execution
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from Device to Host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Results
    FILE *f = fopen("mat-res-fast.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_opt.cu', 'w') as f:
    f.write(cell_str)

In [25]:
### LIBRARY

In [26]:
cell_str=r'''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// Necessary libraries
#include <cuda_runtime.h>
#include <cublas_v2.h>

int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Host allocation
    size_t bytes = n * n * sizeof(float);
    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    // Initialization
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    // Device allocation: cudaMalloc
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    // Data transferring from Host to Device
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // Context creation for the cuBLAS computation
    cublasHandle_t handle;
    cublasCreate(&handle);

    // Parameter setting, as usual
    float alpha = 1.0f;
    float beta = 0.0f;

    // CUDA Timing events
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Start Timing
    cudaEventRecord(start);

    // cuBLAS computation: launch of the function "Sgemm" (Single precision GEneral Matrix Multiply)
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,
                d_a, n,
                &beta,
                d_c, n);

    // Stop Timing
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("cuBLAS Time for n=%d: %.4f seconds\n", n, milliseconds / 1000.0f);

    cublasDestroy(handle);

    // Data transferring (results) from Device to Host
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);


    // Check
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation: both Host and Device
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [27]:
### ADVANCED

In [28]:
cell_str = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

// Parameters
#define BM 128
#define BN 128
#define BK 16
#define TM 8
#define TN 8

// ---- KERNEL CUDA ----
__global__ void sgemm_float4_only(int pad_n, const float *A, const float *B, float *C) {

    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x;
    int by = blockIdx.y;
    int tx = threadIdx.x;
    int ty = threadIdx.y;

    int tid = ty * (BN / TN) + tx; // Da 0 a 255 (256 thread in totale)

    float res[TM][TN] = {0.0f};
    float a_reg[TM];
    float b_reg[TN];

    // Change: loading on the Shared Memory slices through the specific instruction float4
    for (int p = 0; p < (pad_n + BK - 1) / BK; ++p) {

        // Tile A
        for (int i = 0; i < (BM * BK) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int a_row = load_idx / (BK / 4);
            int a_col_v = (load_idx % (BK / 4)) * 4;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col_v;

            // Vectorized Memory Access: 16 byte through only one 128-bit transiction
            float4 tmpA = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_a_row < pad_n && g_a_col < pad_n) {
                tmpA = *(const float4*)&A[g_a_row * pad_n + g_a_col];
            }
            // Vectorized Write in Shared Memory
            *(float4*)&sA[a_row][a_col_v] = tmpA;
        }

        // Tile B
        for (int i = 0; i < (BK * BN) / (256 * 4); ++i) {
            int load_idx = i * 256 + tid;
            int b_row = load_idx / (BN / 4);
            int b_col_v = (load_idx % (BN / 4)) * 4;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col_v;

            // Vectorized Memory Access
            float4 tmpB = make_float4(0.0f, 0.0f, 0.0f, 0.0f);
            if (g_b_row < pad_n && g_b_col < pad_n) {
                tmpB = *(const float4*)&B[g_b_row * pad_n + g_b_col];
            }
            *(float4*)&sB[b_row][b_col_v] = tmpB;
        }

        __syncthreads();

        // Computation on Shared Memory and Registers
        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];

            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }


    // Saving in the Global Memory C
    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < pad_n && c_col < pad_n) {
                C[c_row * pad_n + c_col] = res[i][j];
            }
        }
    }
}


// ---- HOST ----
int main(int argc, char **argv) {

    if (argc < 2) {
        fprintf(stderr, "Error: missing argument in %s \n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);

    // Change: addition of padding logic to handle float4 instructions
    int pad_n = (n + BM - 1) / BM * BM;

    // Host Memory Allocation
    size_t pad_bytes = (size_t)pad_n * pad_n * sizeof(float);
    float *h_a = (float*)malloc(pad_bytes);
    float *h_b = (float*)malloc(pad_bytes);
    float *h_c = (float*)malloc(pad_bytes);

    // Initialization through padding
    for (int i = 0; i < pad_n; i++) {
        for (int j = 0; j < pad_n; j++) {
            if (i < n && j < n) {
                h_a[i * pad_n + j] = 2.0f;
                h_b[i * pad_n + j] = 3.0f;
            } else {
                h_a[i * pad_n + j] = 0.0f;
                h_b[i * pad_n + j] = 0.0f;
            }
            h_c[i * pad_n + j] = 0.0f;
        }
    }

    // Device Memory Allocation
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, pad_bytes);
    cudaMalloc(&d_b, pad_bytes);
    cudaMalloc(&d_c, pad_bytes);

    // Copy from Host to Device
    cudaMemcpy(d_a, h_a, pad_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, pad_bytes, cudaMemcpyHostToDevice);

    // Execution configuration for the Kernel launch: Grid and Block dimensions
    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((pad_n + BN - 1) / BN, (pad_n + BM - 1) / BM);

    // CUDA Event for Timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Kernel Launch + Timing
    cudaEventRecord(start);
    sgemm_float4_only<<<blocksPerGrid, threadsPerBlock>>>(pad_n, d_a, d_b, d_c);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);
    printf("Execution Time: %.4f seconds\n", milliseconds / 1000.0f);

    // Copy from Device to Host
    cudaMemcpy(h_c, d_c, pad_bytes, cudaMemcpyDeviceToHost);

    // Results
    FILE *f = fopen("mat-res-float4.txt", "w");
    if (f) {
        fprintf(f, "%d\n\n", n);
        int limit = (n < 1000) ? n : 1000;
        for (int i = 0; i < limit; i++) {
            for (int j = 0; j < limit; j++) {
                fprintf(f, "%.0f ", h_c[i * pad_n + j]);
            }
            fprintf(f, "\n");
        }
        fclose(f);
    }

    // Memory deallocation
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_adv.cu', 'w') as f:
    f.write(cell_str)

In [29]:
cell_str = r'''#!/bin/bash

# Parameters
SIZES=(10000 15000 20000)
NUM_RUNS=3

echo "Starting the compilation..."

# Compilations
nvcc -arch=sm_75 -O3 cuda_matrixmult_naive.cu -o run_naive
nvcc -arch=sm_75 -O3 cuda_matrixmult_opt.cu -o run_optimized
nvcc -arch=sm_75 -O3 cuda_matrixmult_cuBLAS.cu -o run_cublas -lcublas
nvcc -arch=sm_75 -O3 cuda_matrixmult_adv.cu -o run_float4

echo "Compilation Completed!"

# Starting Execution...

for N in "${SIZES[@]}"; do
    echo ""
    echo ""
    echo " Test: $N x $N Matrix"
    echo ""

    echo ">>> Naive Version"
    for ((i=1; i<=NUM_RUNS; i++)); do
        echo -n "  [Run $i/$NUM_RUNS] "
        ./run_naive $N
    done
    echo "---------------------------------------------------------------------"

    echo ">>> Optimized Version"
    for ((i=1; i<=NUM_RUNS; i++)); do
        echo -n "  [Run $i/$NUM_RUNS] "
        ./run_optimized $N
    done
    echo "---------------------------------------------------------------------"

    echo ">>> Library version: cuBLAS"
    for ((i=1; i<=NUM_RUNS; i++)); do
        echo -n "  [Run $i/$NUM_RUNS] "
        ./run_cublas $N
    done
    echo "---------------------------------------------------------------------"

    echo ">>> Advanced Version"
    for ((i=1; i<=NUM_RUNS; i++)); do
        echo -n "  [Run $i/$NUM_RUNS] "
        ./run_float4 $N
    done
    echo ""
done

echo ""
echo " Benchmark Completed!"
'''

with open('run_benchmark.sh', 'w') as f:
    f.write(cell_str)

In [30]:
!chmod +x run_benchmark.sh
!./run_benchmark.sh

Starting the compilation...
Compilation Completed!


 Test: 10000 x 10000 Matrix

>>> Naive Version
  [Run 1/3] Starting the computation...
Execution Time: 3.4294 seconds
  [Run 2/3] Starting the computation...
Execution Time: 3.4629 seconds
  [Run 3/3] Starting the computation...
Execution Time: 3.4943 seconds
---------------------------------------------------------------------
>>> Optimized Version
  [Run 1/3] Starting the computation...Fast Execution Time: 0.4957 seconds
  [Run 2/3] Starting the computation...Fast Execution Time: 0.4979 seconds
  [Run 3/3] Starting the computation...Fast Execution Time: 0.4966 seconds
---------------------------------------------------------------------
>>> Library version: cuBLAS
  [Run 1/3] cuBLAS Time for n=10000: 0.4451 seconds
  [Run 2/3] cuBLAS Time for n=10000: 0.4490 seconds
  [Run 3/3] cuBLAS Time for n=10000: 0.4478 seconds
---------------------------------------------------------------------
>>> Advanced Version
  [Run 1/3] Execution Tim